<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/main/notebooks/LLaDA_Sudoku_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLaDA Sudoku on Colab Pro

Run the GitHub project with a persistent model copy in Google Drive. Choose **Runtime → Change runtime type → GPU → L4 or A100** (subject to availability), then run the cells in order. Start with one puzzle.

The first session downloads about 16 GB of weights to Drive. Later sessions reuse those files. Each fresh runtime still needs to read the weights into GPU memory. The optional copy to `/content` speeds repeated model loads within a session; it uses runtime disk space, not another Drive copy.

For this **private GitHub repository**, open the notebook through Colab's GitHub tab after authorizing GitHub, or upload this `.ipynb` file to Colab. The clone cell separately needs a fine-grained GitHub token with **Contents: read** access to this repository. Store it in Colab **Secrets** as `GITHUB_TOKEN` and allow notebook access, or enter it into the hidden prompt. The token is never written to the clone URL, repository, or notebook output.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


## 1. Get the code from GitHub
This cell clones the repository, or updates an existing clean checkout without discarding changes.


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "main"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
    existing_url = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if existing_url != REPO_URL:
        raise RuntimeError("The existing checkout belongs to a different repository.")
    git_args = ["-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH]
else:
    git_args = ["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)]

def run_git(token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *git_args], env=env, capture_output=True, text=True)

result = run_git()
if result.returncode:
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        token = getpass.getpass("GitHub token with read access to this private repository: ")
    try:
        result = run_git(token)
    finally:
        del token
if result.returncode:
    raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")
print(f"Code ready: {REPO_DIR}")
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Install dependencies
Keep Colab’s CUDA-enabled PyTorch. Model execution uses a fresh Python process so it sees these installed package versions.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")], check=True)


## 3. Mount Drive and choose the run size
Keep at least **20 GB of free Google Drive storage** for the model and dataset, with extra room for results. Colab Pro and Drive storage are separate allowances.

Keep `NUM_SAMPLES = 1` for verification. Then run the three-puzzle length comparison below and inspect its results before increasing the sample count.

Set `COPY_MODEL_TO_RUNTIME = False` to load directly from Drive. The default keeps the persistent copy on Drive and stages model files on the runtime disk for loading.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
DRIVE_MODEL_DIR = DRIVE_ROOT / "models/LLaDA-8B-Instruct"
RUNTIME_MODEL_DIR = Path("/content/models/LLaDA-8B-Instruct")
COPY_MODEL_TO_RUNTIME = True
NUM_SAMPLES = 1

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
MODEL_PATH = RUNTIME_MODEL_DIR if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIR
print("Persistent files:", DRIVE_ROOT)


## 4. Check the data and write this run’s configuration
The first dataset preparation may process all one million rows before selecting your requested sample. The CSV columns are explicitly strings to preserve leading zeros.


In [ ]:
import yaml

config = yaml.safe_load((REPO_DIR / "configs/sudoku.yaml").read_text())
config["data"].update(num_samples=NUM_SAMPLES, cache_dir=os.environ["HF_DATASETS_CACHE"])
config["model"].update(model_path=str(MODEL_PATH), device="cuda", local_files_only=True)
config["output_dir"] = str(DRIVE_ROOT / "results")
config["provenance"] = {
    "git_commit": subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip(),
    "model_revision": "08b83a6feb34df1a6011b80c3c00c7563e963b07",
}
RUN_CONFIG = Path("/content/sudoku-run.yaml")
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
subprocess.run([sys.executable, "-u", str(REPO_DIR / "Llada-Sudoku/run_sudoku.py"), "--config", str(RUN_CONFIG), "--dataset-only"], check=True)


## 5. Cache the model on Drive
Download the tokenizer, model configuration, Python model implementation, and all six weight shards. A fixed model revision keeps experiments consistent. Hugging Face checks its download metadata and reuses unchanged files on subsequent sessions; keep the model directory's hidden `.cache` folder.

If a download is interrupted, rerun this cell. If it reports a missing file or Drive quota problem, resolve that before proceeding. No model weights are committed to GitHub.


In [ ]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--drive-dir", str(DRIVE_MODEL_DIR)]
if COPY_MODEL_TO_RUNTIME:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


## 6. Verify one puzzle
Keep `NUM_SAMPLES = 1` initially. The model loads once for this invocation, and each puzzle's raw response and metrics are saved to Drive. `status: completed` means inference finished, not that Sudoku was solved.

This cell identifies the newly created run so the inspection cells cannot silently select an older saved result. Interrupted runs retain completed predictions; automatic resume is not implemented.


In [ ]:
# Clear the selected run before starting, including when rerunning this cell.
latest = None
first_record = None
RESULTS_ROOT = Path(config["output_dir"])
previous_runs = set(RESULTS_ROOT.glob("*/metrics.json"))
subprocess.run([sys.executable, "-u", str(REPO_DIR / "Llada-Sudoku/run_sudoku.py"), "--config", str(RUN_CONFIG)], check=True)
new_runs = set(RESULTS_ROOT.glob("*/metrics.json")) - previous_runs
if len(new_runs) != 1:
    raise RuntimeError("Could not identify exactly one new run. Inspect the result folders manually.")
latest = new_runs.pop().parent
print("This run:", latest)


## 7. Inspect the verification result
An unparseable response receives zero exact and cell accuracy. Completion, clue preservation, and Sudoku validity are reported separately. A complete grid can still be wrong.

The chart keeps each line of an incomplete numeric response in its original row. Missing cells remain blank. Complete parsed grids are checked against clues and Sudoku rules; conflicts are flagged without repairing the answer.


In [ ]:
import json

if latest is None:
    raise RuntimeError("Run the verification cell successfully before inspecting its results.")
print("Results:", latest)
print(json.dumps(json.loads((latest / "metrics.json").read_text()), indent=2))
with (latest / "predictions.jsonl").open() as handle:
    first_record = handle.readline()
if not first_record:
    raise RuntimeError("This run has no completed predictions.")
record = json.loads(first_record)
print("First prediction:")
print(json.dumps(record, indent=2))


In [ ]:
import importlib
import matplotlib.pyplot as plt

module_dir = str(REPO_DIR / "Llada-Sudoku")
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
# Reload helpers when the GitHub checkout was updated in an existing runtime.
import run_sudoku
import sudoku_visualization
importlib.reload(run_sudoku)
importlib.reload(sudoku_visualization)

fig = sudoku_visualization.plot_prediction(record)
plt.show()


## 8. Compare 128 and 256 generation tokens
Run this after inspecting the one-puzzle result. Both runs use the **same first three puzzles**, the same prompt, **128 sampling steps**, **32-token blocks**, and unchanged model, temperature, and remasking settings. The comparison always uses three puzzles, regardless of `NUM_SAMPLES`.

The model loads once for both runs. Each run saves its own configuration, raw predictions, and metrics. The comparison saves a puzzle manifest, JSON summary, and CSV table in a new Drive folder. A longer response is **not** assumed to be a correct solution. These six predictions are a diagnostic comparison, not a reliable benchmark.


In [ ]:
from datetime import datetime, timezone

COMPARISON_DIR = DRIVE_ROOT / "comparisons" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
subprocess.run([
    sys.executable, "-u", str(REPO_DIR / "Llada-Sudoku/compare_generation.py"),
    "--config", str(RUN_CONFIG), "--output-dir", str(COMPARISON_DIR),
], check=True)
print("Comparison:", COMPARISON_DIR)


## 9. Inspect the saved comparison
The table reads the current comparison's saved CSV. Rates use all processed puzzles as the denominator, including incomplete answers. Sudoku validity checks rows, columns, and boxes independently of clue preservation; inspect both rates.

Inspect all six raw responses and charts below. Review the saved `predictions.jsonl`, `metrics.json`, and `config.json` files before scaling. If generation still fails to preserve clues or follow Sudoku rules, increasing the sample count only measures that failure more broadly.


In [ ]:
import pandas as pd
from IPython.display import display

report = json.loads((COMPARISON_DIR / "comparison.json").read_text())
print("Comparison status:", report["status"])
print("Saved files:", COMPARISON_DIR)
if not report["runs"]:
    raise RuntimeError("No comparison run finished yet; inspect its partial result folders.")
table = pd.read_csv(COMPARISON_DIR / "comparison.csv")
rate_columns = ["completion_rate", "clue_preservation_rate", "sudoku_validity_rate", "exact_accuracy", "cell_accuracy"]
display(table.drop(columns="run_dir").style.format({name: "{:.1%}" for name in rate_columns}))
for run in report["runs"]:
    print(f"{run['gen_length']} tokens: {run['run_dir']}")


In [ ]:
for run in report["runs"]:
    run_dir = Path(run["run_dir"])
    records = [json.loads(line) for line in (run_dir / "predictions.jsonl").read_text().splitlines()]
    for comparison_record in records:
        print(f"Generation length {run['gen_length']} | puzzle {comparison_record['index']}")
        print("Raw output:")
        print(comparison_record["raw_output"])
        print({name: comparison_record[name] for name in ("parse_success", "clues_preserved", "valid_sudoku", "exact", "correct_cells")})
        fig = sudoku_visualization.plot_prediction(comparison_record)
        plt.show()
        plt.close(fig)


## 10. Increase the sample count only after review
If the setup works and the results justify a larger evaluation, change `NUM_SAMPLES` to 10 or 100 in step 3, rerun step 4 to write the configuration, and rerun steps 6–7. The length comparison remains fixed at the first three puzzles. To use a different generation length for a larger run, change `config["generation"]["gen_length"]` in step 4 before writing `RUN_CONFIG`.

### Notes and references
- This is LLaDA-only evaluation. Sudoku rules validate generated answers; no solver fills, repairs, or replaces them.
- The parser accepts exactly 81 digits in the range 1–9 after removing whitespace, or a standalone contiguous 81-digit answer embedded in text. It does not truncate extra digits or pad partial answers. Unparseable output has an empty `prediction`, zero matching cells, and false validation flags.
- Exact accuracy compares all 81 digits with the reference. Cell accuracy is matching cells divided by `81 × processed puzzles`. Completion, clue-preservation, and Sudoku-validity rates divide their respective passing counts by processed puzzles.
- Raw numeric lines that cannot fit within nine rows of nine columns are shown as text, not reshaped into a misleading grid.
- Saved notebook outputs are cleared in Git. Rerun cells to obtain results from your Colab session.
- [LLaDA](https://github.com/ML-GSAI/LLaDA) · [Model](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct) · [Dataset](https://huggingface.co/datasets/certen/Sudoku-1M-Puzzles)
- [Persistent Hugging Face downloads](https://huggingface.co/docs/huggingface_hub/guides/download#download-files-to-a-local-folder) · [Colab resource and storage FAQ](https://research.google.com/colaboratory/faq.html)
